# ADC target landscape — remote feasibility audit

This notebook validates the public TANGLE TCGA-BRCA manifest and representative tensor formats in an ephemeral Colab runtime. It does **not** copy the pathology dataset to the local computer and does **not** fit a model. Named target expression comes from the public UCSC Xena matrix because TANGLE's 4,999-value RNA tensors do not include a gene-order map.

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import gdown

RUNTIME = Path('/content/adc_target_landscape_audit')
RUNTIME.mkdir(parents=True, exist_ok=True)
print('Ephemeral Colab workspace:', RUNTIME)

In [ ]:
# Download only the small public TANGLE manifest to the Colab runtime.
MANIFEST_ID = '1Yr9pURgLu19xJ51ZV5-AwNCbppUUjuRf'
manifest_path = RUNTIME / 'tcga_brca.csv'
gdown.download(id=MANIFEST_ID, output=str(manifest_path), quiet=False)
manifest = pd.read_csv(manifest_path)
required = {'case_id', 'slide_id'}
assert required.issubset(manifest.columns), manifest.columns.tolist()
print({'slides': len(manifest), 'patients': manifest.case_id.nunique(), 'duplicate_patient_rows': int(manifest.case_id.duplicated(keep=False).sum())})
display(manifest.head())

In [ ]:
# Representative public tensors: format check only; these two files need not be a matched pair.
SAMPLES = {
    'uni': ('1q6mJiuWg35fs2zTOTBB7uUtdEL4ttJhY', 'uni_sample.pt'),
    'anonymous_rna': ('1j-61IF9h-adv6qdTIALgLIzpQHMjOY8h', 'rna_sample.pt'),
}
loaded = {}
for label, (file_id, name) in SAMPLES.items():
    path = RUNTIME / name
    gdown.download(id=file_id, output=str(path), quiet=False)
    loaded[label] = torch.load(path, map_location='cpu', weights_only=False)
    print(label, tuple(loaded[label].shape), loaded[label].dtype)

assert loaded['uni'].ndim == 2 and loaded['uni'].shape[1] == 1024
assert loaded['anonymous_rna'].ndim == 1 and loaded['anonymous_rna'].numel() == 4999
print('PASS: UNI is an (n_patches, 1024) matrix; RNA is an anonymous 4,999-value vector.')

In [ ]:
# Download the named public UCSC Xena matrix into Colab (about 80 MiB compressed).
XENA_URL = 'https://tcga-xena-hub.s3.us-east-1.amazonaws.com/download/TCGA.BRCA.sampleMap%2FHiSeqV2_PANCAN.gz'
xena_path = RUNTIME / 'HiSeqV2_PANCAN.gz'
if not xena_path.exists():
    import urllib.request
    urllib.request.urlretrieve(XENA_URL, xena_path)
expression = pd.read_csv(xena_path, sep='\t', compression='gzip', index_col=0)
primary_cols = [c for c in expression.columns if len(c) >= 15 and c[13:15] == '01']
primary = expression.loc[:, primary_cols]
primary.columns = [c[:12] for c in primary.columns]
primary = primary.T.groupby(level=0).mean().T
manifest_patients = set(manifest.case_id.astype(str).str[:12])
matched = sorted(manifest_patients & set(primary.columns))
missing = sorted(manifest_patients - set(primary.columns))
print({'manifest_patients': len(manifest_patients), 'matched_primary_tumours': len(matched), 'missing_patients': missing})
assert len(matched) == 983 and len(missing) == 1

In [ ]:
# Audit the locked six-target pilot panel using named rows and preserved aliases.
targets = ['ERBB2', 'TACSTD2', 'PVRL4', 'FOLR1', 'SLC39A6', 'MET']
assert set(targets).issubset(expression.index)
audit = primary.loc[targets, matched].T.describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).T
audit['sd'] = primary.loc[targets, matched].T.std()
audit['iqr'] = audit['75%'] - audit['25%']
display(audit[['count', 'mean', 'sd', '50%', 'iqr', '5%', '95%']].round(3))
print('CONDITIONAL GO: formats, identities, named targets, and 983-patient match are reproducible. No modeling was run.')